In [1]:
#| default_exp 30_multihop-datasets

In [2]:
#| export
import pandas as pd, os, json, numpy as np, scipy.sparse as sp, re

from tqdm.auto import tqdm
from typing import Optional, List, Dict

from sugar.core import *

## Helper function

In [3]:
#| export
def load_jsonl(fname):
    data = list()
    with open(fname) as file:
        for line in file: data.append(json.loads(line))
    return data
    

In [4]:
#| export
def load_json(fname:str):
    with open(fname) as f:
        return json.load(f)
        

## `Musique`

In [36]:
#| export
def load_musique_raw(fname:str, title:Optional[List]=None, vocab:Optional[Dict]=None):
    raw_data = load_jsonl(fname)
    
    qry_ids, qry_txt = [], []

    title = [] if title is None else title
    vocab = {} if vocab is None else vocab

    data, indices, indptr = [], [], [0]

    for o in tqdm(raw_data):
        if o["answerable"]:
            qry_ids.append(o["id"])
            qry_txt.append(o["question"])
    
            for lbl in o["paragraphs"]:
                if lbl["paragraph_text"] in vocab:
                    idx = vocab[lbl["paragraph_text"]]
                else:
                    idx = len(vocab)
                    vocab[lbl["paragraph_text"]] = idx
                    title.append(lbl["title"])
    
                if lbl["is_supporting"]:
                    data.append(1.0)
                    indices.append(idx)
                    
            indptr.append(len(indices))
    
    matrix = sp.csr_matrix((data, indices, indptr), shape=(len(indptr) - 1, len(vocab)))
    return qry_ids, qry_txt, title, vocab, matrix
    

In [37]:
#| export
def process_musique(data_dir:str, save_dir:str):
    trn_ids, trn_txt, title, vocab, trn_lbl = load_multihop_raw(f"{data_dir}/musique_ans_v1.0_train.jsonl")
    
    tst_ids, tst_txt, title, vocab, tst_lbl = load_multihop_raw(f"{data_dir}/musique_ans_v1.0_dev.jsonl", 
                                                                title=title, vocab=vocab)
    
    trn_lbl.resize((trn_lbl.shape[0], tst_lbl.shape[1]))

    os.makedirs(f"{save_dir}/raw_data", exist_ok=True)

    save_raw_file(f"{save_dir}/raw_data/train.raw.csv", trn_ids, trn_txt)
    sp.save_npz(f"{save_dir}/trn_X_Y.npz", trn_lbl)

    save_raw_file(f"{save_dir}/raw_data/test.raw.csv", tst_ids, tst_txt)
    sp.save_npz(f"{save_dir}/tst_X_Y.npz", tst_lbl)

    lbl_txt = sorted(vocab, key=lambda x: vocab[x])
    lbl_txt = [f"{p} [SEP] {q}" for p,q in zip(title, lbl_txt)]
    lbl_ids = list(range(len(lbl_txt)))
    save_raw_file(f"{save_dir}/raw_data/label.raw.csv", lbl_ids, lbl_txt)

    return (trn_ids, trn_txt, trn_lbl), (tst_ids, tst_txt, tst_lbl), (lbl_ids, lbl_txt)
    

In [160]:
data_dir = "/Users/suchith720/Projects/data/multihop/musique/original_data"
save_dir = "/Users/suchith720/Projects/data/multihop/musique/XC/"

In [161]:
trn_data, tst_data, lbl_data = process_musique(data_dir, save_dir)

  0%|          | 0/19938 [00:00<?, ?it/s]

  0%|          | 0/2417 [00:00<?, ?it/s]

In [162]:
trn_data[2]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 46613 stored elements and shape (19938, 101962)>

In [163]:
tst_data[2]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6404 stored elements and shape (2417, 101962)>

## `2WikiMultiHopQA`

In [38]:
#| export
def load_2wikimultihopqa_raw(fname:str, vocab:Optional[Dict]=None):
    raw_data = load_json(fname)

    qry_ids, qry_txt = [], []

    title = [] if title is None else title
    vocab = {} if vocab is None else vocab
    
    data, indices, indptr = [], [], [0]
    
    for o in tqdm(raw_data):
        qry_ids.append(o["_id"])
        qry_txt.append(o["question"])
    
        supporting_facts = set([f[0] for f in o["supporting_facts"]])
        labels = [(p, f"{p} [SEP] {' '.join(q)}") for p,q in o["context"]]
        
        for fact, lbl in labels:
            idx = vocab.setdefault(lbl, len(vocab))
        
            if fact in supporting_facts:
                data.append(1.0)
                indices.append(idx)
                
        indptr.append(len(indices))
        
    matrix = sp.csr_matrix((data, indices, indptr), shape=(len(indptr) - 1, len(vocab)))
    return qry_ids, qry_txt, vocab, matrix
    

In [106]:
#| export
def process_2wikimultihopqa(data_dir:str, save_dir:str):
    trn_ids, trn_txt, vocab, trn_lbl = load_2wikimultihopqa_raw(f"{data_dir}/train.json")
    
    tst_ids, tst_txt, vocab, tst_lbl = load_2wikimultihopqa_raw(f"{data_dir}/dev.json", vocab=vocab)
    
    trn_lbl.resize((trn_lbl.shape[0], tst_lbl.shape[1]))

    os.makedirs(f"{save_dir}/raw_data", exist_ok=True)

    save_raw_file(f"{save_dir}/raw_data/train.raw.csv", trn_ids, trn_txt)
    sp.save_npz(f"{save_dir}/trn_X_Y.npz", trn_lbl)

    save_raw_file(f"{save_dir}/raw_data/test.raw.csv", tst_ids, tst_txt)
    sp.save_npz(f"{save_dir}/tst_X_Y.npz", tst_lbl)

    lbl_txt = sorted(vocab, key=lambda x: vocab[x])
    lbl_ids = list(range(len(lbl_txt)))
    save_raw_file(f"{save_dir}/raw_data/label.raw.csv", lbl_ids, lbl_txt)

    return (trn_ids, trn_txt, trn_lbl), (tst_ids, tst_txt, tst_lbl), (lbl_ids, lbl_txt)
    

In [170]:
data_dir = "/Users/suchith720/Projects/data/multihop/2wikimultihopqa/original_data/"
save_dir = "/Users/suchith720/Projects/data/multihop/2wikimultihopqa/XC/"

In [ ]:
process_2wikimultihopqa(data_dir, save_dir)

## `Hover`

In [43]:
data_dir = "/Users/suchith720/Projects/data/multihop/hover/original_data/"
save_dir = "/Users/suchith720/Projects/data/multihop/hover/XC/"

In [14]:
!ls /Users/suchith720/Projects/data/multihop/hover/original_data/

hover_dev_release_v1.1.json   hover_train_release_v1.1.json
hover_test_release_v1.1.json


In [28]:
fname = f"{data_dir}/hover_train_release_v1.1.json"

In [29]:
data = load_json(fname)

In [48]:
data[100]

{'uid': 'cb424d8e-0047-4b13-8385-2d6d13af20fd',
 'claim': "A hockey team calls Madison Square Garden it's home. That team, along with the New York Islanders, and the Brooklyn Nets NHL franchise, are popular in the New York metropolitan area. That team, along with the New York Islanders, and the Brooklyn Nets NHL franchise, are popular in the a part of a city.  Coverage of news on Vos Iz Neias? focuses on this part of a city.",
 'supporting_facts': [['1974–75 New York Islanders season', 2],
  ['New York Rangers', 3],
  ['Madison Square Garden', 5],
  ['Vos Iz Neias?', 3]],
 'label': 'NOT_SUPPORTED',
 'num_hops': 4,
 'hpqa_id': '5a820e8855429926c1cdae14'}

## `Ex-fever`

In [6]:
data_dir = "/Users/suchith720/Projects/data/multihop/exfever/original_data/"
save_dir = "/Users/suchith720/Projects/data/multihop/exfever/XC/"

In [22]:
os.listdir(f"{data_dir}/EX-FEVER/data")

['baseline.png',
 'test.csv',
 'data_base.zip',
 'mini_test.csv',
 'dev.csv',
 'train.csv',
 'length.png',
 'hopps.png',
 'wiki_db.db',
 'example.png']

In [29]:
trn_df = pd.read_csv(f"{data_dir}/EX-FEVER/data/train.csv")
trn_df.head()

,claim,explanation,label,golden entity,mention,result entity,discard number
0,The Hit is the first film Stephen Frears direc...,No information shows that The Hit is the first...,NOT ENOUGH INFO,"['The_Hit_(1984_film)', 'Tim_Roth']",['Tim Roth'],NaN,NaN
1,Gemini is an Indian Tamil-language crime actio...,Gemini is an Indian Tamil-language crime actio...,SUPPORT,"['Gemini_(2002_Tamil_film)', 'AVM_Productions']",['AVM Productions'],NaN,NaN
2,Zach Galifianakis is an American actor who app...,Zach Galifianakis is an American actor who app...,REFUTE,"['Zach_Galifianakis', 'Due_Date']",['Due Date'],NaN,NaN
3,Mark Alan Ruffalo went on to star in a 2002 Am...,Mark Alan Ruffalo went on to star in Shutter I...,REFUTE,"['Mark_Ruffalo', 'Shutter_Island_(film)', 'Shu...","['Shutter Island', '2003 novel of the same name']",NaN,NaN
4,Martin is currently writing the sixth novel in...,"Martin is currently writing the sixth novel, T...",SUPPORT,"['A_Song_of_Ice_and_Fire', 'The_Winds_of_Winter']",['The Winds of Winter'],NaN,NaN


In [30]:
dev_df = pd.read_csv(f"{data_dir}/EX-FEVER/data/dev.csv")
dev_df.head()

,claim,explanation,label,golden entity,mention,result entity,discard number
0,Pineapple Express is a 2008 American stoner co...,Pineapple Express is a 2008 American stoner co...,SUPPORT,"['Pineapple_Express_(film)', 'Judd_Apatow']",['Judd Apatow'],NaN,NaN
1,Doctor Zhivago by George Orwell is a relativel...,"Doctor Zhivago is a novel by Boris Pasternak, ...",REFUTE,"['Doctor_Zhivago_(novel)', 'Novel', 'Narrative']","['novel', 'narrative']",NaN,NaN
2,The Contender is an American film that has a p...,The Contender is a 2000 American political dra...,SUPPORT,"['The_Contender_(2000_film)', 'Political_drama']",['political drama'],NaN,NaN
3,The Premier League Asia Trophy is a biennial p...,The Premier League Asia Trophy is a biennial p...,REFUTE,"['Premier_League_Asia_Trophy', 'FIFA_World_Cup']",['FIFA World Cup'],NaN,NaN
4,Pineapple Express is a 2008 American stoner co...,Pineapple Express is a 2008 American stoner co...,REFUTE,"['Pineapple_Express_(film)', 'David_Gordon_Gre...","['David Gordon Green', 'The Sitter']",NaN,NaN


In [31]:
tst_df = pd.read_csv(f"{data_dir}/EX-FEVER/data/test.csv")
tst_df.head()

,claim,explanation,label,golden entity,mention,result entity,discard number
0,See.SZA.Run is the debut extended play by Cana...,See.SZA.Run is the debut extended play by Amer...,REFUTE,"['See.SZA.Run', 'Miguel_(singer)', 'Wildheart_...","['Miguel', 'Wildheart']","['See.SZA.Run', 'Miguel_(singer)']",3.0
1,Resident Evil is the sixth film written and di...,No information shows that Resident Evil is the...,NOT ENOUGH INFO,"['Resident_Evil_(film)', 'Michelle_Rodriguez',...","['Michelle Rodriguez', 'James Cameron']",NaN,NaN
2,Simon Kunz's notable filmography includes a 19...,Dennis Quaid's notable filmography includes Th...,REFUTE,"['Dennis_Quaid', 'The_Big_Easy_(film)']",['The Big Easy'],NaN,NaN
3,The Walt Disney Company is headquartered at th...,The Walt Disney Company is headquartered in Bu...,NOT ENOUGH INFO,"['The_Walt_Disney_Company', 'Burbank,_Californ...","['Burbank, California', 'Nickelodeon Animation...","['The_Walt_Disney_Company', 'Burbank,_Californ...",3.0
4,Live Your Life is a song by American rapper T....,Live Your Life is a song by American rapper T....,NOT ENOUGH INFO,"['Live_Your_Life_(T.I._song)', 'Paper_Trail', ...","['Paper Trail', 'Lil Wayne']","['Live_Your_Life_(T.I._song)', 'Paper_Trail']",3.0


In [32]:
df.columns

Index(['claim', 'explanation', 'label', 'golden entity', 'mention',
       'result entity', 'discard number'],
      dtype='object')

## `Morehopqa`

In [93]:
#| export
def load_morehopqa_raw(fname:str, vocab:Optional[Dict]=None):
    raw_data = load_json(fname)

    qry_ids, qry_txt = [], []
    vocab = {} if vocab is None else vocab
    
    data, indices, indptr = [], [], [0]
    
    for o in tqdm(raw_data):
        qry_ids.append(o["_id"])
        qry_txt.append(o["question"])
        
        labels = [f"{p} [SEP] {' '.join(q)}" for p,q in o["context"]]
        
        for lbl in labels:
            idx = vocab.setdefault(lbl, len(vocab))
            data.append(1.0)
            indices.append(idx)
                
        indptr.append(len(indices))
        
    matrix = sp.csr_matrix((data, indices, indptr), shape=(len(indptr) - 1, len(vocab)))
    return qry_ids, qry_txt, vocab, matrix
    

In [94]:
#| export
def process_morehopqa(data_dir:str, save_dir:str):
    trn_ids, trn_txt, vocab, trn_lbl = load_morehopqa_raw(f"{data_dir}/without_human_verification.json")
    
    tst_ids, tst_txt, vocab, tst_lbl = load_morehopqa_raw(f"{data_dir}/with_human_verification.json", vocab=vocab)
    
    trn_lbl.resize((trn_lbl.shape[0], tst_lbl.shape[1]))

    os.makedirs(f"{save_dir}/raw_data", exist_ok=True)

    save_raw_file(f"{save_dir}/raw_data/train.raw.csv", trn_ids, trn_txt)
    sp.save_npz(f"{save_dir}/trn_X_Y.npz", trn_lbl)

    save_raw_file(f"{save_dir}/raw_data/test.raw.csv", tst_ids, tst_txt)
    sp.save_npz(f"{save_dir}/tst_X_Y.npz", tst_lbl)

    lbl_txt = sorted(vocab, key=lambda x: vocab[x])
    lbl_ids = list(range(len(lbl_txt)))
    save_raw_file(f"{save_dir}/raw_data/label.raw.csv", lbl_ids, lbl_txt)

    return (trn_ids, trn_txt, trn_lbl), (tst_ids, tst_txt, tst_lbl), (lbl_ids, lbl_txt)
    

In [95]:
data_dir = "/Users/suchith720/Projects/data/multihop/morehopqa/original_data/"
save_dir = "/Users/suchith720/Projects/data/multihop/morehopqa/XC/"

In [97]:
trn_data, tst_data, lbl_data = process_morehopqa(data_dir, save_dir)

  0%|          | 0/2502 [00:00<?, ?it/s]

  0%|          | 0/1118 [00:00<?, ?it/s]

## `MultiHopRAG`

In [154]:
#| export
def load_multihoprag_raw(fname:str, vocab:Dict):
    raw_data = load_json(fname)

    qry_ids, qry_txt = [], []
    
    data, indices, indptr = [], [], [0]
    
    for i,o in tqdm(enumerate(raw_data), total=len(raw_data)):
        qry_ids.append(i)
        qry_txt.append(o["query"])
        
        labels = [k["title"] for k in o["evidence_list"]]
        
        for lbl in labels:
            assert lbl in vocab, f"`{lbl}` not present in the corpus."
            
            idx = vocab[lbl]
            data.append(1.0)
            indices.append(idx)
                
        indptr.append(len(indices))
        
    matrix = sp.csr_matrix((data, indices, indptr), shape=(len(indptr) - 1, len(vocab)))
    return qry_ids, qry_txt, matrix
    

In [155]:
#| export
def process_multihoprag(data_dir:str, save_dir:str):
    labels = load_json(f"{data_dir}/corpus.json")
    vocab = {o["title"]:i for i,o in enumerate(labels)}
    
    tst_ids, tst_txt, tst_lbl = load_multihoprag_raw(f"{data_dir}/MultiHopRAG.json", vocab)
    
    os.makedirs(f"{save_dir}/raw_data", exist_ok=True)

    save_raw_file(f"{save_dir}/raw_data/test.raw.csv", tst_ids, tst_txt)
    sp.save_npz(f"{save_dir}/tst_X_Y.npz", tst_lbl)

    lbl_txt = [f"{o['title']} [SEP] {o['author']} [SEP] {o['source']} [SEP] {o['category']} [SEP] {o['body']}" for o in labels]
    lbl_ids = list(range(len(lbl_txt)))
    save_raw_file(f"{save_dir}/raw_data/label.raw.csv", lbl_ids, lbl_txt)

    return (tst_ids, tst_txt, tst_lbl), (lbl_ids, lbl_txt)
    

In [156]:
data_dir = "/Users/suchith720/Projects/data/multihop/multihoprag/original_data/"
save_dir = "/Users/suchith720/Projects/data/multihop/multihoprag/XC/"

In [157]:
tst_data, lbl_data = process_multihoprag(data_dir, save_dir)

  0%|          | 0/2556 [00:00<?, ?it/s]

In [115]:
fname = f"/Users/suchith720/Projects/data/multihop/multihoprag/original_data/MultiHopRAG.json"
data = load_json(fname)

In [123]:
data[100]

{'query': "Does the 'Fortune' article suggest that 'Fifth Third 1.67% Card' holders can redeem points for statement credits in the same way that 'Fortune' indicates 'American Express Cash Magnet® cardholders' can redeem cash back rewards for statement credits?",
 'answer': 'Yes',
 'question_type': 'comparison_query',
 'evidence_list': [{'title': 'Fifth Third 1.67% Cash/Back Card: One of the simplest cash back credit cards with an above-average return',
   'author': 'Joseph Hostetler',
   'url': 'https://fortune.com/recommends/credit-cards/fifth-third-1-67-cash-back-card-review/',
   'source': 'Fortune',
   'category': 'business',
   'published_at': '2023-11-13T22:03:50+00:00',
   'fact': 'When you earn points with the Fifth Third 1.67% Card, you can use them toward a statement credit on your card to offset your balance.'},
  {'title': 'American Express Cash Magnet® Card review: a simple, no-nonsense cash back card',
   'author': 'Kat Tretina',
   'url': 'https://fortune.com/recommends/

In [118]:
fname = f"/Users/suchith720/Projects/data/multihop/multihoprag/original_data/corpus.json"
corpus = load_json(fname)

In [125]:
corpus_titles = set([o["title"] for o in corpus])

In [124]:
corpus[10]

{'title': 'SBF’s trial starts soon, but how did he — and FTX — get here?',
 'author': 'Jacquelyn Melinek',
 'source': 'TechCrunch',
 'published_at': '2023-10-01T14:00:29+00:00',
 'category': 'technology',
 'url': 'https://techcrunch.com/2023/10/01/ftx-lawsuit-timeline/',
 'body': 'SBF’s trial has started, this is how he and FTX got here\n\nThe highly anticipated criminal trial for Sam Bankman-Fried, former CEO of bankrupt crypto exchange FTX, started Tuesday to determine whether he’s guilty of seven counts of fraud and conspiracy. And as one former federal prosecutor put it: “The odds seem to be stacked against him at this point.”\n\nThe 31-year-old co-founded FTX in 2019; within a few years the once third-largest crypto exchange’s valuation hit $32 billion at its peak. It’s now trying to claw back any funds to distribute to creditors.\n\nBut how did the once third-largest crypto exchange get here?\n\nBefore FTX, Bankman-Fried co-founded crypto-trading firm Alameda Research in 2017. He